# Step 2 - Clean, split and lock the test set  (4 - 17 Oct)
Objectives 1 and 2: clean, remove duplicates (including URL hosts that appear more than once or in both classes), split 70/10/20 with a fixed seed, and put the test set aside untouched.

> Re-running this notebook rebuilds the split deterministically (same seed). It refuses to run after the final test.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)
from toadwatch import config as C
from toadwatch.pipeline import run_data_preparation
from toadwatch.splitting import verify_test_untouched
# After the final test the split is frozen: re-compute in memory only, never overwrite the files.
out = run_data_preparation(save=not C.FINAL_TEST_LOCK.exists())

## What the cleaning removed

In [2]:
import json
print(json.dumps(out['calls_report'], indent=2, default=str))
print(json.dumps(out['msgs_report'], indent=2, default=str))

{
  "rows_raw": 8952,
  "unusable_rows": 54,
  "conflicting_label_rows": 0,
  "duplicate_rows": 1783,
  "near_duplicate_rows": 404,
  "rows_clean": 6711,
  "class_counts": {
    "0": 4666,
    "1": 2045
  }
}
{
  "rows_raw": 167759,
  "unusable_rows": 9,
  "url_duplicate_hosts": {
    "conflicting_label_rows": 477,
    "duplicate_rows": 45284
  },
  "text_duplicates": {
    "conflicting_label_rows": 4,
    "duplicate_rows": 986
  },
  "rows_clean": 120153,
  "counts_by_kind_label": {
    "email|0": 9641,
    "email|1": 9677,
    "sms|0": 8935,
    "sms|1": 1253,
    "url|0": 47307,
    "url|1": 43340
  }
}


## Split sizes (70 / 10 / 20)

In [3]:
import json
json.loads(C.SPLIT_MANIFEST_JSON.read_text())['files'] and pd.DataFrame(json.loads(C.SPLIT_MANIFEST_JSON.read_text())['files']).T

,rows,fraud,safe,sha256
train,2862,1432,1430,f7ccde8bba661673f8a65b641d24704fc388055b5d1b48...
val,410,204,206,76a8f5204bc7eb94664fa6868acaab3b930e471f877008...
test,818,409,409,a0389db9296552e644b13a75cc845ee5f2d64fd9c86fc3...


## Leak checks (all must print True)

In [4]:
from toadwatch.cleaning import host_of
rec = out['records']
splits = {s: rec[rec.split == s] for s in ['train', 'val', 'test']}
print('call ids disjoint     :', all(not set(splits[a].call_id) & set(splits[b].call_id) for a, b in [('train','val'),('train','test'),('val','test')]))
print('message ids disjoint  :', all(not set(splits[a].item_id) & set(splits[b].item_id) for a, b in [('train','val'),('train','test'),('val','test')]))
url_hosts = {s: set(d[d.msg_kind == 'url'].msg_url.map(host_of)) for s, d in splits.items()}
print('URL hosts disjoint    :', all(not url_hosts[a] & url_hosts[b] for a, b in [('train','val'),('train','test'),('val','test')]))
print('call texts unique     :', rec.call_text.is_unique)
print('test file fingerprint :', verify_test_untouched())

call ids disjoint     : True
message ids disjoint  : True
URL hosts disjoint    : True
call texts unique     : True
test file fingerprint : True


## Near-duplicate check (train vs validation, TF-IDF cosine > 0.9)
Robocall scripts are replayed with tiny wording changes, so exact de-duplication is not enough; near-duplicates are removed in `clean_calls`. Both shares should be ~0 (the test set is deliberately not inspected).

In [5]:
from toadwatch.cleaning import near_duplicate_share
print('calls   :', round(near_duplicate_share(splits['train'].call_text, splits['val'].call_text), 3))
print('messages:', round(near_duplicate_share(splits['train'].msg_text, splits['val'].msg_text), 3))

calls   : 0.0


messages: 0.027


In [6]:
rec.groupby(['split', 'label', 'msg_kind']).size().unstack(fill_value=0)

msg_kind     email  sms  url
split label                 
test  0        136  137  136
      1        136  137  136
train 0        477  477  476
      1        477  477  478
val   0         69   68   69
      1         68   68   68